In [1]:
# Self implementation of Decision Tree

In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [19]:
# File setup
fpath_test = 'datasets/kaggle_titanic_train.csv'
fpath_train = 'datasets/kaggle_titanic_test.csv'

df_test = pd.read_csv(fpath_train)
df_train = pd.read_csv(fpath_test)
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [24]:
# Data exploration
target_var = 'Survived'
print(f'n_train = {df_train.shape}')

print(f'== Number of unique values per field ==')
for column in df_train.columns:
    print(f'{column}: {df_train[column].value_counts().size}')

print(f'Fields that are likely to be too numerous for classification')
drop_fields = [column for column in df_train.columns if 
               df_train[column].value_counts().size > df_train.shape[0]/4]

drop_fields += ['Cabin']
print(drop_fields)

# Missing Values
print(f'\nCounting number of empty values in each field')
print(df_train.isna().sum())

n_train = (891, 12)
== Number of unique values per field ==
PassengerId: 891
Survived: 2
Pclass: 3
Name: 891
Sex: 2
Age: 88
SibSp: 7
Parch: 7
Ticket: 681
Fare: 248
Cabin: 147
Embarked: 3
Fields that are likely to be too numerous for classification
['PassengerId', 'Name', 'Ticket', 'Fare', 'Cabin']

Counting number of empty values in each field
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [5]:
# Removing fields that are likely unique / too numerous to be useful for classification
df_train.drop(columns=drop_fields, inplace=True)
df_train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Cabin,Embarked
0,0,3,male,22.0,1,0,NaN,S
1,1,1,female,38.0,1,0,C85,C
2,1,3,female,26.0,0,0,NaN,S
3,1,1,female,35.0,1,0,C123,S
4,0,3,male,35.0,0,0,NaN,S


In [6]:
# Testing for class imbalance
print(df_train[target_var].value_counts())

Survived
0    549
1    342
Name: count, dtype: int64


In [7]:
# Calculating gini
def gini_calc(target, feature_split: list):
    ginis, node_counts = np.zeros((2, len(feature_split)))
    
    for i, split in enumerate(feature_split):
        targets_in_node = target[split]
        node_counts[i] = len(targets_in_node)
        ginis[i] = 1 - np.array(
                        [(np.where(targets_in_node==c)[0].shape[0]/node_counts[i])**2
                         for c in targets_in_node.unique()]
                    ).sum()
    
    return (ginis*(node_counts/len(target))).sum()

In [21]:
# Calculating minimum gini for a given feature
def feature_split(target, feature):

    for i, split in enumerate(feature.unique()):
        print(split)
    
    return None

feature_split(df_train[target_var], df_train['Embarked'])

S
C
Q
nan


In [25]:
min_gini = np.array([
    gini_calc(
        df_train[target_var],
        [df_train[col]==c]
    )
    for col in df_train.columns
    for c in df_train[col].unique()
])